# 01 — Data Understanding

**Business question:** Which e-commerce customers are likely to churn, why, and how can nature-inspired algorithms (GA / PSO) improve prediction?

This notebook frames the data-mining problem, documents each feature, checks data quality, and saves a clean interim snapshot. No modelling yet.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")
print(f"Raw data     : {DATA_RAW}")
print(f"Interim      : {DATA_INTERIM}")
print(f"Processed    : {DATA_PROCESSED}")
print(f"Figures      : {FIGURES_DIR}")
print(f"Models       : {MODELS_DIR}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42
Raw data     : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw
Interim      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim
Processed    : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\processed
Figures      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\reports\figures
Models       : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-custome

## 1. Load raw Excel

The Kaggle workbook has two sheets: `Data Dict` (definitions) and `E Comm` (customer rows).

In [2]:
import pandas as pd

RAW_XLSX = DATA_RAW / "E_Commerce_Dataset.xlsx"
assert RAW_XLSX.exists(), f"Missing dataset: {RAW_XLSX}"

data_dict_raw = pd.read_excel(RAW_XLSX, sheet_name="Data Dict", header=None)
df = pd.read_excel(RAW_XLSX, sheet_name="E Comm")

print("E Comm shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

E Comm shape: (5630, 20)
Columns: ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


## 2. Feature dictionary

We rebuild the dictionary from the Data Dict sheet and add short business notes that matter for churn analysis.

In [3]:
# Parse Data Dict sheet (header row is embedded)
dd = data_dict_raw.copy()
# Find the header-like row
header_idx = dd.index[dd.apply(lambda r: r.astype(str).str.contains("Variable", na=False).any(), axis=1)][0]
dd2 = dd.iloc[header_idx + 1 :, [1, 2, 3]].copy()
dd2.columns = ["Table", "Variable", "Description"]
dd2 = dd2.dropna(subset=["Variable"]).reset_index(drop=True)

business_notes = {
    "CustomerID": "Surrogate key — drop from models; use only for joins / examples.",
    "Churn": "Target (1 = churned, 0 = retained). Class is imbalanced.",
    "Tenure": "Relationship length. Short tenure often correlates with higher churn risk.",
    "PreferredLoginDevice": "Channel preference (Phone / Mobile Phone / Computer). May need harmonisation.",
    "CityTier": "Urbanisation tier (ordinal 1–3). Proxy for market maturity / logistics.",
    "WarehouseToHome": "Delivery distance. Longer distance can hurt experience.",
    "PreferredPaymentMode": "Payment preference. Overlapping labels (CC vs Credit Card, COD vs Cash on Delivery).",
    "Gender": "Demographic attribute.",
    "HourSpendOnApp": "Engagement proxy. Missingness is common.",
    "NumberOfDeviceRegistered": "Multi-device usage — engagement / account complexity.",
    "PreferedOrderCat": "Category affinity (note spelling Prefered). Mobile vs Mobile Phone overlap.",
    "SatisfactionScore": "Ordinal service score (1–5). Key retention signal.",
    "MaritalStatus": "Demographic segment.",
    "NumberOfAddress": "Address book size — possible multi-location / switching signal.",
    "Complain": "Binary complaint flag last month — strong churn risk signal.",
    "OrderAmountHikeFromlastYear": "Spend growth. Declining hike may signal disengagement.",
    "CouponUsed": "Promo sensitivity.",
    "OrderCount": "Recent purchase frequency.",
    "DaySinceLastOrder": "Recency. High values = dormant risk (but watch leakage vs churn definition).",
    "CashbackAmount": "Reward intensity. Low cashback + complaints may amplify churn.",
}

feat_dict = dd2.copy()
feat_dict["BusinessNote"] = feat_dict["Variable"].map(business_notes)
feat_dict.to_csv(DATA_INTERIM / "feature_dictionary.csv", index=False)
feat_dict

,Table,Variable,Description,BusinessNote
0,E Comm,CustomerID,Unique customer ID,Surrogate key — drop from models; use only for...
1,E Comm,Churn,Churn Flag,"Target (1 = churned, 0 = retained). Class is i..."
2,E Comm,Tenure,Tenure of customer in organization,Relationship length. Short tenure often correl...
3,E Comm,PreferredLoginDevice,Preferred login device of customer,Channel preference (Phone / Mobile Phone / Com...
4,E Comm,CityTier,City tier,Urbanisation tier (ordinal 1–3). Proxy for mar...
5,E Comm,WarehouseToHome,Distance in between warehouse to home of customer,Delivery distance. Longer distance can hurt ex...
6,E Comm,PreferredPaymentMode,Preferred payment method of customer,Payment preference. Overlapping labels (CC vs ...
7,E Comm,Gender,Gender of customer,Demographic attribute.
8,E Comm,HourSpendOnApp,Number of hours spend on mobile application or...,Engagement proxy. Missingness is common.
9,E Comm,NumberOfDeviceRegistered,Total number of deceives is registered on part...,Multi-device usage — engagement / account comp...


## 3. Schema, missingness, duplicates

In [4]:
print("Dtypes:")
print(df.dtypes)
print("\nMissing counts:")
missing = df.isna().sum()
print(missing[missing > 0])
print(f"\nRows with any NA: {df.isna().any(axis=1).sum()} ({df.isna().any(axis=1).mean():.1%})")
print(f"CustomerID unique: {df['CustomerID'].nunique()} / {len(df)}")
print(f"Duplicate CustomerID rows: {df['CustomerID'].duplicated().sum()}")
print(f"Fully duplicate rows: {df.duplicated().sum()}")

Dtypes:
CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice               str
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode               str
Gender                             str
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                   str
SatisfactionScore                int64
MaritalStatus                      str
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
dtype: object

Missing counts:
Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderC

**Data-quality notes**

- Seven numeric behavioural columns have missing values (~4–5% each). We will impute in preprocessing (median), not drop rows wholesale.
- No duplicate `CustomerID`s — one row per customer.
- Categorical label overlaps (`Phone`/`Mobile Phone`, `CC`/`Credit Card`, `COD`/`Cash on Delivery`, `Mobile`/`Mobile Phone` in order category) should be harmonised before encoding.

## 4. Target distribution and metric strategy

In [5]:
churn_rate = df["Churn"].mean()
counts = df["Churn"].value_counts().sort_index()
print(counts)
print(f"Churn rate: {churn_rate:.2%} ({counts.get(1, 0)} / {len(df)})")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Retained (0)", "Churned (1)"], counts.values, color=["#2a9d8f", "#e76f51"])
ax.set_ylabel("Customers")
ax.set_title("Churn class balance")
for i, v in enumerate(counts.values):
    ax.text(i, v + 40, str(v), ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_churn_balance.png", dpi=150)
plt.show()

Churn
0    4682
1     948
Name: count, dtype: int64
Churn rate: 16.84% (948 / 5630)


C:\Users\ishan\AppData\Local\Temp\ipykernel_20936\1300375082.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Why not Accuracy as the primary metric?**

With ~16.8% churn, a trivial “always retain” classifier is already ~83% accurate. For retention teams:

- **False Negative** (miss a churner) = lost customer + wasted lifetime value.
- **False Positive** (flag a loyal customer) = unnecessary discount / outreach cost.

We therefore prioritise **Recall**, **F1**, and **PR-AUC**, reporting Accuracy only as a secondary sanity check.

## 5. Save interim snapshot

In [6]:
out_csv = DATA_INTERIM / "ecomm_validated.csv"
df.to_csv(out_csv, index=False)

meta = {
    "n_rows": int(len(df)),
    "n_cols": int(df.shape[1]),
    "churn_rate": float(churn_rate),
    "n_missing_cells": int(df.isna().sum().sum()),
    "duplicate_customer_ids": int(df["CustomerID"].duplicated().sum()),
}
import json
(DATA_INTERIM / "ecomm_validated_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved", out_csv)
print(meta)

Saved C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\ecomm_validated.csv
{'n_rows': 5630, 'n_cols': 20, 'churn_rate': 0.16838365896980462, 'n_missing_cells': 1856, 'duplicate_customer_ids': 0}


## Exit checklist

- Feature dictionary written to `data/interim/feature_dictionary.csv`
- Interim customer table written to `data/interim/ecomm_validated.csv`
- Class imbalance and metric priorities documented

**Next:** Notebook `02` — deep exploratory analysis and clustering.